# Install Dependencies

In [1]:
!pip install python-docx joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.2 MB/s eta 0:00:00


# Imports

In [2]:
import os
import gc
import json
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from docx import Document
from datetime import datetime

# Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# Path Setup

In [4]:
RAW_BASE = "/users/"
BASE_SAVE_DIR = "/users/"

DATASET_NAME = "CIC_IIoT_2025"
SAVE_DIR = os.path.join(BASE_SAVE_DIR, DATASET_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print("📂 RAW:", RAW_BASE)
print("💾 SAVE:", SAVE_DIR)

📂 RAW: /users/
💾 SAVE: /users/


# Load Dataset (X + y)

In [5]:
X_path = os.path.join(RAW_BASE, "X_preprocessed.csv")
y_path = os.path.join(RAW_BASE, "y_resampled.csv")

dfX = pd.read_csv(X_path)
dfY = pd.read_csv(y_path)

df = pd.concat([dfX, dfY], axis=1)
original_shape = df.shape

print("Loaded CIC-IIoT-2025:", df.shape)
df.head()

Loaded CIC-IIoT-2025: (42035, 51)


,log_data-types_count,log_messages_count,network_fragmentation-score,network_fragmented-packets,network_interval-packets,network_ip-flags_avg,network_ip-flags_min,network_ip-length_max,network_ip-length_std_deviation,network_ips_all_count,...,network_ttl_std_deviation,network_window-size_avg,network_window-size_max,network_window-size_min,network_window-size_std_deviation,device_edge1,device_mqtt-broker,mac_dc:a6:32:dc:27:d4,mac_dc:a6:32:dc:28:46,Attack_Category_Encoded
0,-0.587994,-0.521089,-0.33825,-0.337513,-0.872600,0.437634,-0.267393,-1.005079,-0.679642,0.123709,...,-0.866616,0.798540,0.784914,-1.110477,1.454877,1,0,1,0,1
1,-0.587994,-0.521089,-0.33825,-0.337513,-0.872691,0.411035,-0.267393,-1.005079,-0.681462,0.123709,...,-0.866616,0.792984,0.784914,-1.110477,1.440477,1,0,1,0,1
2,-0.587994,-0.521089,-0.33825,-0.337513,-0.872735,0.405362,-0.267393,-1.005079,-0.681859,-0.014872,...,-0.866616,0.791727,0.784914,-1.110477,1.437205,1,0,1,0,1
3,-0.587994,-0.521089,-0.33825,-0.337513,-0.872735,0.369013,-0.267393,-1.005079,-0.684235,0.123709,...,-0.866616,0.784163,0.784914,-1.110477,1.417347,1,0,1,0,1
4,-0.587994,-0.521089,-0.33825,-0.337513,-0.872735,0.363241,-0.267393,-1.005079,-0.684672,0.123709,...,-0.866616,0.782799,0.784914,-1.110477,1.413757,1,0,1,0,1


# Detect Label Column

In [6]:
label_candidates = [
    "Label",
    "Attack_Category_Encoded",
    "Attack_Category",
    "Attack",
    "Attack_Type",
]

label_col = None
for c in label_candidates:
    if c in df.columns:
        label_col = c
        break

if label_col is None:
    raise ValueError("❌ No label column found in CIC-IIoT-2025 dataset.")

print("✔ Using label column:", label_col)

df.rename(columns={label_col: "Label"}, inplace=True)
df["Label"] = df["Label"].astype(int)

✔ Using label column: Attack_Category_Encoded


# Encode Labels + Store Class Names

In [7]:
label_encoder = LabelEncoder()
df["Label_encoded"] = label_encoder.fit_transform(df["Label"])

classes = list(label_encoder.classes_)
num_classes = len(classes)

label_encoder_path = os.path.join(SAVE_DIR, "label_encoder.pkl")
joblib.dump(label_encoder, label_encoder_path)

print("✔ Classes:", classes)
print("💾 Saved Label Encoder →", label_encoder_path)

✔ Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
💾 Saved Label Encoder → /users/


# Numeric Feature Selection

In [8]:
X_raw = df.drop(columns=["Label", "Label_encoded"])
y = df["Label_encoded"].values

all_features_before = list(X_raw.columns)
total_features_before = len(all_features_before)

numeric_cols = X_raw.select_dtypes(include=["int64", "float64", "float32"]).columns.tolist()
X_num = X_raw[numeric_cols].copy()

print("🔹 TOTAL FEATURES BEFORE:", total_features_before)
print("🔹 NUMERIC FEATURES:", len(numeric_cols))

🔹 TOTAL FEATURES BEFORE: 50
🔹 NUMERIC FEATURES: 50


# Clean & Scale

In [9]:
X_num = X_num.replace([np.inf, -np.inf], np.nan)
X_num = X_num.fillna(X_num.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num.values)

scaler_path = os.path.join(SAVE_DIR, "scaler.pkl")
joblib.dump(scaler, scaler_path)

print("✔ Scaled shape:", X_scaled.shape)
print("💾 Scaler saved:", scaler_path)

selected_feature_names = numeric_cols
num_selected_features = len(numeric_cols)

✔ Scaled shape: (42035, 50)
💾 Scaler saved: /users/


# Stratified Train/Val/Test Split (70/15/15)

In [10]:
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss1.split(X_scaled, y))

X_train, X_temp = X_scaled[train_idx], X_scaled[temp_idx]
y_train, y_temp = y[train_idx], y[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))

X_val, X_test = X_temp[val_idx], X_temp[test_idx]
y_val, y_test = y_temp[val_idx], y_temp[test_idx]

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

Train: (29424, 50)
Val  : (6305, 50)
Test : (6306, 50)


# Class Distribution

In [11]:
def count_classes(arr):
    ct = Counter(arr.tolist())
    return {str(k): int(v) for k, v in ct.items()}

train_dist = count_classes(y_train)
val_dist = count_classes(y_val)
test_dist = count_classes(y_test)

print("Train Class Dist:", train_dist)
print("Val Class Dist:", val_dist)
print("Test Class Dist:", test_dist)

Train Class Dist: {'1': 4204, '3': 4203, '4': 4203, '5': 4204, '0': 4204, '2': 4203, '6': 4203}
Val Class Dist: {'3': 901, '0': 901, '2': 901, '5': 900, '6': 901, '4': 901, '1': 900}
Test Class Dist: {'1': 901, '6': 901, '3': 901, '0': 900, '2': 901, '5': 901, '4': 901}


# Autoencoder Model

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = X_train.shape[1]
latent_dim = 64

In [13]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

In [14]:
ae = AE().to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Train Autoencoder

In [15]:
train_tensor = torch.tensor(X_train, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=1024, shuffle=True)

In [16]:
EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    ae.train()
    total_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out, z = ae(batch)
        loss = criterion(out, batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}/{EPOCHS} — Loss: {total_loss/len(train_loader):.6f}")

Epoch 1/20 — Loss: 0.668989
Epoch 2/20 — Loss: 0.243755
Epoch 3/20 — Loss: 0.145942
Epoch 4/20 — Loss: 0.089639
Epoch 5/20 — Loss: 0.060625
Epoch 6/20 — Loss: 0.046209
Epoch 7/20 — Loss: 0.037295
Epoch 8/20 — Loss: 0.031204
Epoch 9/20 — Loss: 0.026739
Epoch 10/20 — Loss: 0.023245
Epoch 11/20 — Loss: 0.020555
Epoch 12/20 — Loss: 0.018656
Epoch 13/20 — Loss: 0.016817
Epoch 14/20 — Loss: 0.016060
Epoch 15/20 — Loss: 0.014114
Epoch 16/20 — Loss: 0.012933
Epoch 17/20 — Loss: 0.012029
Epoch 18/20 — Loss: 0.011620
Epoch 19/20 — Loss: 0.010892
Epoch 20/20 — Loss: 0.010243


# Extract Latent Features

In [17]:
def encode(model, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        _, Z = model(X_t)
    return Z.cpu().numpy()

z_train = encode(ae, X_train)
z_val   = encode(ae, X_val)
z_test  = encode(ae, X_test)

print("Latent Shapes:", z_train.shape, z_val.shape, z_test.shape)

Latent Shapes: (29424, 64) (6305, 64) (6306, 64)


# Save Outputs

In [18]:
np.save(os.path.join(SAVE_DIR, "train_latent.npy"), z_train)
np.save(os.path.join(SAVE_DIR, "val_latent.npy"),   z_val)
np.save(os.path.join(SAVE_DIR, "test_latent.npy"),  z_test)

np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)
np.save(os.path.join(SAVE_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(SAVE_DIR, "y_test.npy"),  y_test)

with open(os.path.join(SAVE_DIR, "feature_list.txt"), "w") as f:
    for col in selected_feature_names:
        f.write(col + "\n")

torch.save(ae.state_dict(), os.path.join(SAVE_DIR, "autoencoder.pth"))

print("✔ All files saved.")

✔ All files saved.


# JSON Summary

In [19]:
summary = {
    "dataset_name": DATASET_NAME,
    "generated_at": datetime.now().isoformat(),
    "raw": {
        "num_samples": int(original_shape[0]),
        "num_features_total": int(original_shape[1]),
        "feature_names_total": all_features_before,
    },
    "numeric_features": {
        "count": num_selected_features,
        "names": selected_feature_names,
    },
    "classes": {
        "num_classes": num_classes,
        "class_mapping": {int(i): str(cls) for i, cls in enumerate(classes)},
    },
    "splits": {
        "train": {"samples": int(len(y_train)), "class_counts": train_dist},
        "val":   {"samples": int(len(y_val)), "class_counts": val_dist},
        "test":  {"samples": int(len(y_test)), "class_counts": test_dist},
    },
    "latent_dim": latent_dim,
}

json_path = os.path.join(SAVE_DIR, "preprocessing_summary.json")
with open(json_path, "w") as f:
    json.dump(summary, f, indent=4)

print("✔ JSON summary saved:", json_path)

✔ JSON summary saved: /users/


# DOCX Summary

In [20]:
doc = Document()
doc.add_heading(f"Dataset Preprocessing Summary — {DATASET_NAME}", level=1)

doc.add_paragraph(f"Generated at: {datetime.now()}")

doc.add_heading("1. Raw Dataset Overview", level=2)
doc.add_paragraph(f"Total rows: {original_shape[0]}")
doc.add_paragraph(f"Total columns: {original_shape[1]}")

doc.add_heading("2. Numeric Features", level=2)
doc.add_paragraph(f"Selected numeric features: {num_selected_features}")

for col in selected_feature_names[:50]:
    doc.add_paragraph(f"- {col}", style="List Bullet")

doc.add_heading("3. Classes", level=2)
doc.add_paragraph(f"Total classes: {num_classes}")

table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text = "Class Index"
table.rows[0].cells[1].text = "Class Name"
for i, cls in enumerate(classes):
    row = table.add_row().cells
    row[0].text = str(i)
    row[1].text = str(cls)

doc.add_heading("4. Splits", level=2)
doc.add_paragraph(f"Train size: {len(y_train)}")
doc.add_paragraph(f"Val size: {len(y_val)}")
doc.add_paragraph(f"Test size: {len(y_test)}")

doc_path = os.path.join(SAVE_DIR, "preprocessing_summary.docx")
doc.save(doc_path)
print("✔ DOCX saved:", doc_path)

✔ DOCX saved: /users/
